# Multiplying and Dividing Polynomials

Notebook 15 covered adding and subtracting polynomials. Notebook 17 covered multiplying and dividing single terms (monomials). This notebook combines both: how to multiply and divide full polynomials.

The core mechanism is the **distributive property** — every term in one polynomial is multiplied by every term in the other, and the results are collected by degree. For division, there are two cases: dividing by a monomial (straightforward) and dividing by a polynomial (an algorithm called long division).

**Prerequisites:** Polynomial Operations — Adding and Subtracting (15), Exponent Multiplication and Division (16), Multiplying and Dividing Monomials (17).

In [ ]:
import sys
sys.path.insert(0, '../..')  # repo root, so `shared` is importable
from fractions import Fraction

# poly_str, poly_eval, poly_clean (15) and mono_str (17) were taught in earlier notebooks — imported here, not re-taught
from shared.polynomials import poly_str, poly_eval, poly_clean, mono_str

# Quick smoke test
p = {3: 2, 1: -3, 0: 5}
print(f'Example: {poly_str(p)}  evaluated at x=2: {poly_eval(p, 2)}')

---

## 1. Multiplying a Polynomial by a Monomial

The **distributive property** says $a(b + c) = ab + ac$. A monomial multiplied by a polynomial distributes over every term:

$$3x^2 \cdot (2x^3 + 5x - 4)$$

The monomial $3x^2$ "visits" each term and multiplies it using the monomial rules from notebook 17:

$$= (3x^2 \cdot 2x^3) + (3x^2 \cdot 5x) + (3x^2 \cdot (-4))$$

$$= 6x^5 + 15x^3 - 12x^2$$

Each multiplication: coefficient × coefficient, exponent + exponent. No combining of unlike terms needed here — each product lands on a unique degree.

**Programmer bridge:** it is a `map` over the polynomial's terms:
```python
{exp + mono_exp: coeff * mono_coeff  for exp, coeff in poly.items()}
```
Every term gets the same transformation applied to it.

In [ ]:
def poly_mul_mono(poly, mono_coeff, mono_exp):
    """Multiply a polynomial by a monomial."""
    return {exp + mono_exp: coeff * mono_coeff for exp, coeff in poly.items()}

def show_mul_mono(poly, mc, me):
    print(f'  {mono_str(mc, me)} * ({poly_str(poly)})')
    print(f'  Distribute:')
    for exp in sorted(poly, reverse=True):
        c = poly[exp]
        r_c = mc * c
        r_e = me + exp
        print(f'    {mono_str(mc, me)} * {mono_str(c, exp)} = {mono_str(r_c, r_e)}')
    result = poly_mul_mono(poly, mc, me)
    print(f'  Result: {poly_str(result)}')
    x = 2
    mono_val = mc * (x ** me)
    orig   = mono_val * poly_eval(poly, x)
    check  = poly_eval(result, x)
    print(f'  Verify at x=2: {orig} = {check}  {"✓" if orig == check else "✗"}')
    print()

show_mul_mono({3: 2, 1: 5, 0: -4},  3, 2)   #  3x^2 * (2x^3 + 5x - 4)
show_mul_mono({2: 1, 1: -3, 0: 2}, -2, 1)   # -2x   * (x^2 - 3x + 2)
show_mul_mono({2: 4, 0: -6},        1, 0)   #     1 * (4x^2 - 6)  — multiply by a constant

---

## 2. Multiplying Two Binomials

When both factors are polynomials, apply the distributive property twice — once for each term in the first polynomial.

$$(x + 3)(x + 4)$$

Treat $(x + 3)$ as the monomial-like thing being distributed. The first factor distributes over the second:

$$= x \cdot (x + 4) + 3 \cdot (x + 4)$$

Now distribute each monomial:

$$= (x^2 + 4x) + (3x + 12)$$

Combine like terms:

$$= x^2 + 7x + 12$$

This is often called **FOIL** (First, Outer, Inner, Last) — a mnemonic for the four products when multiplying two binomials. But FOIL is not a rule; it is just a label for the four products that the distributive property produces:

| Label | Product | From |
|-------|---------|------|
| **F**irst | $x \cdot x = x^2$ | first terms of each factor |
| **O**uter | $x \cdot 4 = 4x$ | outer terms |
| **I**nner | $3 \cdot x = 3x$ | inner terms |
| **L**ast | $3 \cdot 4 = 12$ | last terms of each factor |

Sum: $x^2 + 4x + 3x + 12 = x^2 + 7x + 12$.

> FOIL only works for binomial × binomial. The general approach below works for any polynomials.

In [ ]:
def show_foil(p1, p2):
    """Show the four FOIL products for two binomials."""
    assert len(p1) == 2 and len(p2) == 2, 'show_foil expects binomials'
    terms1 = sorted(p1.items(), reverse=True)   # [(exp, coeff), ...] high -> low
    terms2 = sorted(p2.items(), reverse=True)
    labels = ['First', 'Outer', 'Inner', 'Last']
    pairs  = [
        (terms1[0], terms2[0]),
        (terms1[0], terms2[1]),
        (terms1[1], terms2[0]),
        (terms1[1], terms2[1]),
    ]
    print(f'  ({poly_str(p1)})({poly_str(p2)})')
    print()
    accumulated = {}
    for label, ((e1, c1), (e2, c2)) in zip(labels, pairs):
        r_e = e1 + e2
        r_c = c1 * c2
        print(f'  {label:<6}: {mono_str(c1, e1)} * {mono_str(c2, e2)} = {mono_str(r_c, r_e)}')
        accumulated[r_e] = accumulated.get(r_e, 0) + r_c
    result = {k: v for k, v in accumulated.items() if v != 0}
    print(f'  Combine like terms: {poly_str(result)}')
    x = 3
    orig  = poly_eval(p1, x) * poly_eval(p2, x)
    check = poly_eval(result, x)
    print(f'  Verify at x=3: {orig} = {check}  {"✓" if orig == check else "✗"}')
    print()

show_foil({1: 1, 0:  3}, {1: 1, 0:  4})   # (x + 3)(x + 4)
show_foil({1: 2, 0: -1}, {1: 3, 0:  5})   # (2x - 1)(3x + 5)
show_foil({1: 1, 0: -4}, {1: 1, 0:  4})   # (x - 4)(x + 4)  — difference of squares preview

---

## 3. General Polynomial Multiplication

FOIL is a special case of a universal rule: **every term in $P_1$ is multiplied by every term in $P_2$**, and then all products that share the same degree are summed.

$$\text{coeff of } x^n \text{ in the product} = \sum_{i+j=n} a_i \cdot b_j$$

where $a_i$ is the coefficient of $x^i$ in $P_1$ and $b_j$ is the coefficient of $x^j$ in $P_2$.

**Programmer bridge — this is convolution.** The coefficient array of the product is the discrete convolution of the coefficient arrays of the factors. `numpy.polymul` does exactly this.

In code, it is a nested loop:

```python
for each term (e1, c1) in P1:
    for each term (e2, c2) in P2:
        accumulate c1*c2 at degree e1+e2
```

Every pair contributes one product. Pairs that land on the same degree are combined as like terms.

In [ ]:
def poly_multiply(p1, p2):
    """Multiply two polynomials — nested loop over all term pairs."""
    result = {}
    for e1, c1 in p1.items():
        for e2, c2 in p2.items():
            exp   = e1 + e2
            coeff = c1 * c2
            result[exp] = result.get(exp, 0) + coeff
    return poly_clean(result)

# Verify against numpy
import numpy as np

def poly_to_np(p):
    """Convert {exp: coeff} dict to numpy coefficient array (highest degree first)."""
    if not p:
        return np.array([0])
    max_exp = max(p)
    arr = np.zeros(max_exp + 1)
    for exp, coeff in p.items():
        arr[max_exp - exp] = coeff
    return arr

def np_to_poly(arr):
    """Convert numpy coefficient array back to {exp: coeff} dict."""
    n = len(arr) - 1
    return {n - i: float(c) for i, c in enumerate(arr) if c != 0}

examples = [
    ({1: 1, 0:  3}, {1: 1, 0: 4}),              # (x+3)(x+4)      = x^2+7x+12
    ({2: 1, 1:  2, 0: 1}, {1: 1, 0: -1}),       # (x^2+2x+1)(x-1) = x^3+x^2-x-1
    ({2: 2, 1: -3, 0: 1}, {2: 1, 1: 4, 0: -2}), # (2x^2-3x+1)(x^2+4x-2)
    ({1: 1, 0: 1},        {2: 1, 1: -1, 0: 1}), # (x+1)(x^2-x+1) = x^3+1
]

print(f'  {"Expression":<40}  {"Our result":<25}  numpy match')
print(f'  {"-"*40}  {"-"*25}  -----------')
for p1, p2 in examples:
    our   = poly_multiply(p1, p2)
    np_r  = np_to_poly(np.polymul(poly_to_np(p1), poly_to_np(p2)))
    match = all(abs(our.get(e, 0) - np_r.get(e, 0)) < 1e-9
                for e in set(our) | set(np_r))
    expr  = f'({poly_str(p1)})({poly_str(p2)})'
    print(f'  {expr:<40}  {poly_str(our):<25}  {"✓" if match else "✗"}')

---

## 4. Special Products

Three patterns appear so often that they are worth recognising on sight. They are not separate rules — they fall straight out of the general multiplication algorithm. Knowing them saves work.

### Square of a binomial — sum

$$(a + b)^2 = (a + b)(a + b) = a^2 + ab + ab + b^2 = a^2 + 2ab + b^2$$

The middle term is always $2ab$ — twice the product of the two terms. Not $a^2 + b^2$. The cross-term is the most common mistake.

### Square of a binomial — difference

$$(a - b)^2 = a^2 - 2ab + b^2$$

Same pattern, middle term negative.

### Difference of squares

$$(a + b)(a - b) = a^2 - ab + ab - b^2 = a^2 - b^2$$

The middle terms cancel completely — the inner and outer products are equal and opposite. You multiply two binomials and get a binomial: the degree-1 term vanishes.

> These three show up constantly: in factoring, in calculus (derivatives of products), in the quadratic formula, and in Fourier analysis. Recognising them is faster than running the general algorithm every time.

In [ ]:
def verify_special(label, formula, p1, p2, a, b):
    """Confirm a special product formula against the general algorithm."""
    computed = poly_multiply(p1, p2)
    x = 5   # test at x=5 for good separation
    orig  = poly_eval(p1, x) * poly_eval(p2, x)
    check = poly_eval(computed, x)
    print(f'  {label}')
    print(f'    Formula:  {formula}')
    print(f'    Expanded: {poly_str(computed)}')
    print(f'    Verify at x=5 (a={a}, b={b}): {orig} = {check}  {"✓" if orig == check else "✗"}')
    print()

# (a + b)^2 with a=x, b=3  =>  x^2 + 6x + 9
verify_special(
    '(x + 3)^2  [sum squared]',
    'a^2 + 2ab + b^2 = x^2 + 6x + 9',
    {1: 1, 0: 3}, {1: 1, 0: 3}, 'x', 3
)

# (a - b)^2 with a=2x, b=5  =>  4x^2 - 20x + 25
verify_special(
    '(2x - 5)^2  [difference squared]',
    'a^2 - 2ab + b^2 = 4x^2 - 20x + 25',
    {1: 2, 0: -5}, {1: 2, 0: -5}, '2x', 5
)

# (a + b)(a - b) with a=x, b=7  =>  x^2 - 49
verify_special(
    '(x + 7)(x - 7)  [difference of squares]',
    'a^2 - b^2 = x^2 - 49',
    {1: 1, 0: 7}, {1: 1, 0: -7}, 'x', 7
)

# A trap: (x + 3)^2 != x^2 + 9
wrong  = {2: 1, 0: 9}   # x^2 + 9
right  = poly_multiply({1: 1, 0: 3}, {1: 1, 0: 3})
print(f'  Common mistake: (x+3)^2 = {poly_str(right)},  NOT {poly_str(wrong)}')
print(f'  The cross-term 2*x*3 = 6x is always there.')

---

## 5. Dividing a Polynomial by a Monomial

Division by a monomial is the distributive property run in reverse — each term of the polynomial is divided by the monomial independently:

$$\frac{6x^4 - 9x^3 + 12x^2}{3x^2}$$

Split the fraction over the terms:

$$= \frac{6x^4}{3x^2} - \frac{9x^3}{3x^2} + \frac{12x^2}{3x^2}$$

Apply the monomial division rule (from notebook 17) to each term independently:

$$= 2x^2 - 3x + 4$$

**Programmer bridge:** it is a `map` over the polynomial's terms — the inverse of the `map` used when multiplying by a monomial:

```python
{exp - mono_exp: coeff / mono_coeff  for exp, coeff in poly.items()}
```

This only works cleanly when the monomial divides every term. When it does not divide evenly (for example, $\frac{x^2 + 1}{x}$), the result contains a fractional term — a rational expression, not a polynomial.

In [ ]:
def poly_div_mono(poly, mono_coeff, mono_exp):
    """Divide a polynomial by a monomial."""
    result = {}
    for exp, coeff in poly.items():
        r_exp   = exp - mono_exp
        r_coeff = Fraction(coeff, mono_coeff)
        result[r_exp] = r_coeff
    return poly_clean(result)

def show_div_mono(poly, mc, me):
    print(f'  ({poly_str(poly)}) / {mono_str(mc, me)}')
    print(f'  Split over terms:')
    for exp in sorted(poly, reverse=True):
        c = poly[exp]
        r_e = exp - me
        r_c = Fraction(c, mc)
        r_c_disp = int(r_c) if r_c.denominator == 1 else r_c
        print(f'    {mono_str(c, exp)} / {mono_str(mc, me)} = {mono_str(r_c_disp, r_e)}')
    result = poly_div_mono(poly, mc, me)
    print(f'  Result: {poly_str(result)}')
    x = 2
    mono_val = mc * (x ** me)
    orig  = Fraction(poly_eval(poly, x), mono_val)
    check = poly_eval(result, x)
    print(f'  Verify at x=2: {orig} = {check}  {"✓" if orig == check else "✗"}')
    print()

show_div_mono({4:  6, 3: -9, 2: 12}, 3, 2)  # (6x^4 - 9x^3 + 12x^2) / 3x^2
show_div_mono({3: 10, 2:  5, 1: -15}, 5, 1) # (10x^3 + 5x^2 - 15x)  / 5x
show_div_mono({4:  8, 2: -4, 0:  12}, 4, 0) # (8x^4 - 4x^2 + 12)    / 4  (constant divisor)

---

## 6. Dividing a Polynomial by a Polynomial — Long Division

When the divisor is itself a polynomial, the process is **polynomial long division** — the same algorithm as integer long division, applied to terms ordered by degree.

**The algorithm:**
1. Write both polynomials in standard form (descending degree).
2. Divide the **leading term** of the dividend by the **leading term** of the divisor. This gives the next term of the quotient.
3. Multiply that quotient term by the entire divisor.
4. Subtract the result from the current dividend.
5. Repeat from step 2 with the remainder until the remainder's degree is less than the divisor's degree.

**Example:** $(x^2 + 5x + 6) \div (x + 2)$

| Step | Action | Working |
|------|--------|---------|
| 1 | $x^2 \div x = x$ | first quotient term |
| 2 | $x \cdot (x + 2) = x^2 + 2x$ | multiply |
| 3 | $(x^2 + 5x + 6) - (x^2 + 2x) = 3x + 6$ | subtract, remainder |
| 4 | $3x \div x = 3$ | next quotient term |
| 5 | $3 \cdot (x + 2) = 3x + 6$ | multiply |
| 6 | $(3x + 6) - (3x + 6) = 0$ | remainder is 0 |

**Quotient:** $x + 3$, **Remainder:** $0$. Check: $(x + 3)(x + 2) = x^2 + 5x + 6$ ✓

**Programmer bridge:** this is `divmod` for polynomials. At each iteration, you find what to multiply the divisor by to cancel the current leading term — exactly as you find each digit of a quotient in integer division. The degree shrinks by at least 1 each iteration, so the loop always terminates.

In [ ]:
def poly_divmod(dividend, divisor):
    """Polynomial long division. Returns (quotient, remainder) as poly dicts."""
    remainder = {k: Fraction(v) for k, v in dividend.items()}
    quotient  = {}

    deg_d  = max(divisor)
    lead_d = Fraction(divisor[deg_d])

    while remainder and max(remainder) >= deg_d:
        deg_r  = max(remainder)
        lead_r = remainder[deg_r]

        q_coeff = lead_r / lead_d
        q_exp   = deg_r - deg_d

        quotient[q_exp] = quotient.get(q_exp, Fraction(0)) + q_coeff

        for exp, coeff in divisor.items():
            new_exp = exp + q_exp
            remainder[new_exp] = remainder.get(new_exp, Fraction(0)) - q_coeff * coeff

        remainder = {k: v for k, v in remainder.items() if v != 0}

    return poly_clean(quotient), poly_clean(remainder)


def show_poly_divmod(dividend, divisor):
    print(f'  ({poly_str(dividend)}) / ({poly_str(divisor)})')
    print()

    remainder = {k: Fraction(v) for k, v in dividend.items()}
    quotient_terms = []
    deg_d  = max(divisor)
    lead_d = Fraction(divisor[deg_d])
    step   = 1

    while remainder and max(remainder) >= deg_d:
        deg_r  = max(remainder)
        lead_r = remainder[deg_r]
        q_c    = lead_r / lead_d
        q_e    = deg_r - deg_d
        q_c_d  = int(q_c) if q_c.denominator == 1 else q_c

        print(f'  Step {step}: leading term {poly_str(dict(remainder))} -> divide {mono_str(int(lead_r) if lead_r.denominator==1 else lead_r, deg_r)} by {mono_str(int(lead_d) if lead_d.denominator==1 else lead_d, deg_d)} = {mono_str(q_c_d, q_e)}')

        sub_dict = {}
        for exp, coeff in divisor.items():
            se = exp + q_e
            sc = q_c * coeff
            sub_dict[se] = sc

        print(f'         Multiply {mono_str(q_c_d, q_e)} * ({poly_str(divisor)}) = {poly_str(poly_clean(sub_dict))}')

        for exp, coeff in divisor.items():
            new_exp = exp + q_e
            remainder[new_exp] = remainder.get(new_exp, Fraction(0)) - q_c * coeff

        remainder = {k: v for k, v in remainder.items() if v != 0}
        rem_str   = poly_str(poly_clean(dict(remainder))) if remainder else '0'
        print(f'         Remainder after subtract: {rem_str}')
        print()

        quotient_terms.append((q_e, q_c))
        step += 1

    q_dict  = poly_clean({e: c for e, c in quotient_terms})
    rem_str = poly_str(poly_clean(dict(remainder))) if remainder else '0'
    print(f'  Quotient:  {poly_str(q_dict)}')
    print(f'  Remainder: {rem_str}')

    # Verify: quotient * divisor + remainder == dividend (at x=3)
    x = 3
    r_dict   = poly_clean(dict(remainder))
    check    = poly_eval(q_dict, x) * poly_eval(divisor, x) + poly_eval(r_dict, x)
    expected = poly_eval(dividend, x)
    print(f'  Verify at x=3: Q*D + R = {check},  dividend = {expected}  {"✓" if check == expected else "✗"}')
    print()


show_poly_divmod({2: 1, 1:  5, 0:  6}, {1: 1, 0:  2})  # (x^2+5x+6)  / (x+2)
show_poly_divmod({3: 1, 2: -6, 1: 11, 0: -6}, {1: 1, 0: -2})  # (x^3-6x^2+11x-6) / (x-2)

In [ ]:
# Division with a non-zero remainder
print('Division with a remainder:')
show_poly_divmod({2: 1, 1: 3, 0: 1}, {1: 1, 0: 2})  # (x^2+3x+1) / (x+2)  -> x+1, remainder -1

# The sum of cubes pattern: (x^3 - 1) / (x - 1) = x^2 + x + 1
print('Sum/difference of cubes pattern:')
show_poly_divmod({3: 1, 0: -1}, {1: 1, 0: -1})        # (x^3 - 1) / (x - 1)

---

## 7. Worked Examples

### Example 1 — Monomial × polynomial

$$-2x \cdot (3x^2 - x + 5)$$

Distribute $-2x$ over each term:

$$= (-2x)(3x^2) + (-2x)(-x) + (-2x)(5) = -6x^3 + 2x^2 - 10x$$

### Example 2 — Binomial × binomial with coefficients

$$(2x - 3)(4x + 1)$$

F: $2x \cdot 4x = 8x^2$. O: $2x \cdot 1 = 2x$. I: $-3 \cdot 4x = -12x$. L: $-3 \cdot 1 = -3$.

Combine: $8x^2 + (2 - 12)x - 3 = 8x^2 - 10x - 3$.

### Example 3 — Trinomial × binomial

$$(x^2 - 2x + 3)(x + 4)$$

Distribute $x + 4$ over every term of the trinomial:

$$= x^2(x + 4) - 2x(x + 4) + 3(x + 4)$$
$$= (x^3 + 4x^2) + (-2x^2 - 8x) + (3x + 12)$$
$$= x^3 + (4-2)x^2 + (-8+3)x + 12 = x^3 + 2x^2 - 5x + 12$$

### Example 4 — Special product in a larger expression

$$(x + 5)^2 - (x - 5)^2$$

Expand each square using the pattern:

$$= (x^2 + 10x + 25) - (x^2 - 10x + 25)$$

Subtract (distribute the $-$):

$$= x^2 + 10x + 25 - x^2 + 10x - 25 = 20x$$

The $x^2$ and constant terms all cancel. The expression simplifies to a monomial.

### Example 5 — Polynomial ÷ monomial, then verify

$$\frac{12x^4 - 8x^3 + 4x^2}{4x^2} = 3x^2 - 2x + 1$$

Check by multiplying back: $4x^2 \cdot (3x^2 - 2x + 1) = 12x^4 - 8x^3 + 4x^2$ ✓

### Example 6 — Polynomial ÷ polynomial, remainder

$$\frac{2x^3 + x^2 - 5x + 2}{x + 2}$$

Long division gives quotient $2x^2 - 3x + 1$, remainder $0$. So $x + 2$ is a factor of $2x^3 + x^2 - 5x + 2$.

In [ ]:
x = 3

examples = [
    # (label, computed, expected)
    ('Ex 1: -2x(3x^2-x+5) = -6x^3+2x^2-10x',
      poly_eval(poly_mul_mono({2: 3, 1: -1, 0: 5}, -2, 1), x),
      poly_eval({3: -6, 2: 2, 1: -10}, x)),

    ('Ex 2: (2x-3)(4x+1) = 8x^2-10x-3',
      poly_eval(poly_multiply({1: 2, 0: -3}, {1: 4, 0: 1}), x),
      poly_eval({2: 8, 1: -10, 0: -3}, x)),

    ('Ex 3: (x^2-2x+3)(x+4) = x^3+2x^2-5x+12',
      poly_eval(poly_multiply({2: 1, 1: -2, 0: 3}, {1: 1, 0: 4}), x),
      poly_eval({3: 1, 2: 2, 1: -5, 0: 12}, x)),

    ('Ex 4: (x+5)^2-(x-5)^2 = 20x',
      poly_eval(poly_multiply({1:1,0:5},{1:1,0:5}), x)
      - poly_eval(poly_multiply({1:1,0:-5},{1:1,0:-5}), x),
      20 * x),

    ('Ex 5: (12x^4-8x^3+4x^2)/4x^2 = 3x^2-2x+1',
      poly_eval(poly_div_mono({4: 12, 3: -8, 2: 4}, 4, 2), x),
      poly_eval({2: 3, 1: -2, 0: 1}, x)),
]

print(f'Verifying examples at x={x}:')
print()
for label, computed, expected in examples:
    ok = computed == expected
    print(f'  {label}')
    print(f'    computed={computed},  expected={expected}  {"✓" if ok else "✗ MISMATCH"}')
    print()

# Example 6: long division
print('Ex 6: (2x^3+x^2-5x+2) / (x+2)')
q, r = poly_divmod({3: 2, 2: 1, 1: -5, 0: 2}, {1: 1, 0: 2})
print(f'  Quotient: {poly_str(q)},  Remainder: {poly_str(r) if r else "0"}')
check = poly_eval(q, x) * poly_eval({1:1,0:2}, x) + (poly_eval(r,x) if r else 0)
print(f'  Verify Q*(x+2)+R = {check},  original = {poly_eval({3:2,2:1,1:-5,0:2}, x)}  {"✓" if check == poly_eval({3:2,2:1,1:-5,0:2},x) else "✗"}')

---

## 8. Visualisation — The Multiplication Grid

Polynomial multiplication is a Cartesian product of terms. Every pair of terms contributes one cell in a grid. Terms that land on the same degree are like-coloured — they will be combined.

Below: the grid for $(x^2 + 2x + 1)(x + 3)$. Each cell shows the product of one term from each factor. Cells with the same colour share a degree and will be summed.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

p1 = {2: 1, 1: 2, 0: 1}   # x^2 + 2x + 1
p2 = {1: 1, 0: 3}          # x + 3

terms1 = sorted(p1.items(), reverse=True)  # [(2,1), (1,2), (0,1)]
terms2 = sorted(p2.items(), reverse=True)  # [(1,1), (0,3)]

# Colour by resulting degree
degree_colors = {
    3: '#89dceb',   # cyan
    2: '#a6e3a1',   # green
    1: '#fab387',   # orange
    0: '#f38ba8',   # red
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f'({poly_str(p1)})({poly_str(p2)}) — multiplication grid', fontsize=13, fontweight='bold')

# ---- Left: the grid ----
ax = axes[0]
ax.set_xlim(-0.5, len(terms2) - 0.5)
ax.set_ylim(-0.5, len(terms1) - 0.5)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Each cell = one pair of terms', fontsize=11)

# Column headers (p2 terms)
for j, (e2, c2) in enumerate(terms2):
    ax.text(j, len(terms1) - 0.1, mono_str(c2, e2), ha='center', va='bottom', fontsize=12, fontweight='bold')

# Row headers (p1 terms)
for i, (e1, c1) in enumerate(terms1):
    ax.text(-0.6, len(terms1) - 1 - i, mono_str(c1, e1), ha='right', va='center', fontsize=12, fontweight='bold')

for i, (e1, c1) in enumerate(terms1):
    for j, (e2, c2) in enumerate(terms2):
        deg = e1 + e2
        coeff = c1 * c2
        color = degree_colors.get(deg, '#cdd6f4')
        row = len(terms1) - 1 - i
        rect = mpatches.FancyBboxPatch((j - 0.45, row - 0.45), 0.9, 0.9,
                                        boxstyle='round,pad=0.05',
                                        facecolor=color, edgecolor='white', linewidth=2)
        ax.add_patch(rect)
        ax.text(j, row + 0.12, mono_str(coeff, deg), ha='center', va='center', fontsize=11, fontweight='bold')
        ax.text(j, row - 0.22, f'deg {deg}', ha='center', va='center', fontsize=8, color='#555')

# Legend
patches = [mpatches.Patch(color=c, label=f'degree {d}') for d, c in sorted(degree_colors.items(), reverse=True)]
ax.legend(handles=patches, loc='lower right', fontsize=9, framealpha=0.9)

# ---- Right: the result — terms by degree ----
ax2 = axes[1]
product = poly_multiply(p1, p2)

# Build per-degree contribution list
degree_contribs = {}
for e1, c1 in terms1:
    for e2, c2 in terms2:
        d = e1 + e2
        degree_contribs.setdefault(d, []).append(c1 * c2)

degrees = sorted(degree_contribs, reverse=True)
bar_colors = [degree_colors.get(d, '#cdd6f4') for d in degrees]
totals = [sum(degree_contribs[d]) for d in degrees]
deg_labels = [f'x^{d}' if d > 1 else ('x' if d == 1 else '1') for d in degrees]

bars = ax2.bar(deg_labels, totals, color=bar_colors, edgecolor='white', linewidth=2)
for bar, deg, total in zip(bars, degrees, totals):
    contribs = '+'.join(str(c) for c in degree_contribs[deg])
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             f'{contribs}={total}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax2.set_ylabel('Coefficient', fontsize=11)
ax2.set_title(f'Result: {poly_str(product)}', fontsize=11)
ax2.set_ylim(0, max(totals) + 1.8)
ax2.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---

## Exercises

Work each on paper first, then verify in code.

**Multiplication**
1. Expand $4x(x^2 - 3x + 2)$.
2. Expand $(x - 5)(x + 5)$. What kind of special product is this?
3. Expand $(3x - 2)^2$. Write the middle term carefully.
4. Expand $(x + 2)(x^2 - 2x + 4)$. The result is a well-known pattern — what is it?
5. Expand $(2x + 1)(3x^2 - x + 2)$.
6. Simplify $(x + 3)^2 - (x - 3)^2$. Predict the result type before expanding.

**Division**
7. Divide $\dfrac{15x^3 - 10x^2 + 5x}{5x}$.
8. Divide $(x^2 - 9) \div (x - 3)$. Predict the result using the difference-of-squares pattern before computing.
9. Divide $(x^3 + 2x^2 - x - 2) \div (x + 1)$.
10. Divide $(2x^2 + 5x + 1) \div (x + 2)$. What is the remainder?

In [ ]:
# Verify your answers here.
# poly_multiply(p1, p2)          — multiply two polynomials
# poly_mul_mono(poly, coeff, exp) — multiply poly by a monomial
# poly_div_mono(poly, coeff, exp) — divide poly by a monomial
# poly_divmod(dividend, divisor) — polynomial long division -> (quotient, remainder)
# poly_str(p)                    — pretty-print
# poly_eval(p, x)                — evaluate at x

# Example: exercise 2
# p = poly_multiply({1: 1, 0: -5}, {1: 1, 0: 5})
# print(poly_str(p))

---

## Formal Notation

Let $P(x) = \sum_{i=0}^{m} a_i x^i$ and $Q(x) = \sum_{j=0}^{n} b_j x^j$ be polynomials of degrees $m$ and $n$.

**Product:** $\deg(PQ) = m + n$. The coefficient of $x^k$ in $PQ$ is:
$$[x^k]\, PQ = \sum_{i+j=k} a_i b_j$$
This is the **discrete convolution** of the coefficient sequences $(a_i)$ and $(b_j)$.

**Division with remainder:** For any polynomials $P$ (dividend) and $D$ (divisor, $D \neq 0$), there exist unique polynomials $Q$ (quotient) and $R$ (remainder) such that:
$$P = Q \cdot D + R \qquad \text{where } \deg(R) < \deg(D)$$
This is the **polynomial division algorithm** — the direct analogue of the integer division theorem ($a = qb + r$, $0 \leq r < b$).

**Special products** for $a, b \in \mathbb{R}$:

| Pattern | Identity |
|---------|----------|
| Sum squared | $(a + b)^2 = a^2 + 2ab + b^2$ |
| Difference squared | $(a - b)^2 = a^2 - 2ab + b^2$ |
| Difference of squares | $(a + b)(a - b) = a^2 - b^2$ |
| Sum of cubes | $(a + b)(a^2 - ab + b^2) = a^3 + b^3$ |
| Difference of cubes | $(a - b)(a^2 + ab + b^2) = a^3 - b^3$ |

---

## Real-World Connection

- **Digital signal processing — FIR filters:** a finite impulse response filter is defined by a polynomial in $z$ (the z-transform). Cascading two filters means multiplying their polynomials — the output filter has degree equal to the sum of the input degrees. Every audio plugin, image blur, and equaliser you have ever used runs polynomial multiplication at its core.

- **Fast multiplication — FFT-based algorithms:** multiplying two $n$-digit numbers is equivalent to polynomial multiplication. The naive approach is $O(n^2)$ — a nested loop over all pairs. The Fast Fourier Transform reduces this to $O(n \log n)$ by performing polynomial multiplication in the frequency domain. This is how large-integer arithmetic libraries and modern CPUs accelerate multiplication.

- **Error-correcting codes:** Reed-Solomon codes (used in QR codes, CDs, DVDs, and deep-space communication) represent data as polynomial coefficients over finite fields and use polynomial division to detect and correct errors. The remainder from polynomial division encodes the parity check.

- **Computer algebra systems:** SymPy, Mathematica, and Wolfram Alpha implement `poly_multiply` and `poly_divmod` internally — the same nested-loop convolution and long-division algorithm from this notebook, generalised to multivariate polynomials and arbitrary coefficient rings.

- **Kinematics:** if position is a polynomial in time $t$ (e.g., $p(t) = 5t^2 + 3t$) and you want to know total distance as a function of two time endpoints $t_1$ and $t_2$, you compute $p(t_2) - p(t_1)$, which involves polynomial arithmetic. Multiplying two polynomial expressions for combined motion produces a higher-degree polynomial — the product rule applied to functions of time.

---

## Summary

**Multiplication** is the distributive property applied systematically:
- **Monomial × polynomial:** the monomial distributes over every term — a `map` operation: `{exp + e: coeff * c for exp, coeff in poly.items()}`.
- **Polynomial × polynomial:** every term in $P_1$ pairs with every term in $P_2$ — a nested loop. Products that share a degree are summed as like terms. This is discrete convolution of the coefficient sequences.
- **Special products** — $(a+b)^2$, $(a-b)^2$, $(a+b)(a-b)$ — are patterns that fall out of the general algorithm. Recognise them to save steps.

**Division** has two cases:
- **÷ monomial:** split the fraction over each term, apply the monomial quotient rule to each independently. A `map` — the inverse of multiplying by a monomial.
- **÷ polynomial (long division):** iteratively cancel the leading term of the remainder by finding the right quotient term. Identical in structure to integer `divmod`. Always produces a unique quotient $Q$ and remainder $R$ with $\deg(R) < \deg(D)$.

**Verification:** evaluate both sides at any $x$. For division: check that $Q \cdot D + R$ equals the original dividend.